# 07 TT-rank / Bond Dimension Sweep：3-core TT-matrix の保存量・圧縮率・再構成誤差

## 今回の到達点

今回は、dense weight

$$
W\in\mathbb R^{8\times 8}
$$

に対して tensorization を固定し、**共通内部 rank の上限だけ**を変えます。

確認する関係は次の2つです。

$$
\boxed{
\text{rank}
\rightarrow
\text{parameter count}
\rightarrow
\text{compression ratio}
}
$$

$$
\boxed{
\text{rank}
\rightarrow
\text{weight reconstruction error}
}
$$

今回は **weight の比較だけ**で止めます。Neural Network の prediction / accuracy には進みません。

### 今回追加しない内容

- Fashion-MNIST / CIFAR-10
- ground-truth label
- prediction agreement
- accuracy / certified accuracy
- fine-tuning
- training / backward
- tensorization 候補の比較
- 不均一 rank 探索
- 数値的 rank 判定による自動 rank 選択
- Pareto frontier
- DMRG

中央の演習セルには **TODO と `_____`** を残します。別セルに解答は置きません。


## 1. 実行環境

小さな再構成誤差を確認するため `torch.float64` を使います。

- device: CPU
- dtype: `torch.float64`
- seed: 固定


In [52]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import math
import pandas as pd
import torch
from torch.testing import assert_close

torch.manual_seed(7)

dtype = torch.float64
device = torch.device("cpu")

rtol = 1e-10
atol = 1e-12


## 2. Tensorization を固定する

### 何のための処理か

rank の影響だけを比較するため、全実験で同じ tensorization を使います。

$$
m_{\text{modes}}=(m_1,m_2,m_3)=(2,2,2),
$$

$$
n_{\text{modes}}=(n_1,n_2,n_3)=(2,2,2).
$$

したがって、

$$
m=m_1m_2m_3=8,
\qquad
n=n_1n_2n_3=8.
$$

各 physical pair のサイズを

$$
s_k=m_kn_k
$$

とすると、今回は

$$
(s_1,s_2,s_3)=(4,4,4)
$$

です。

### `m_modes`

1. **意味**: output 側の tensorization
2. **数式上の定義**: $(m_1,m_2,m_3)$
3. **shape**: Python tuple、長さ3
4. **各軸の意味**: 第1・第2・第3 core の output physical mode
5. **何から計算されるか**: 今回は固定値 `(2, 2, 2)`
6. **用途**: dense row index を3つの TT physical mode に分ける

### `n_modes`

1. **意味**: input 側の tensorization
2. **数式上の定義**: $(n_1,n_2,n_3)$
3. **shape**: Python tuple、長さ3
4. **各軸の意味**: 第1・第2・第3 core の input physical mode
5. **何から計算されるか**: 今回は固定値 `(2, 2, 2)`
6. **用途**: dense column index を3つの TT physical mode に分ける


In [53]:
m_modes = (2, 2, 2)
n_modes = (2, 2, 2)

m1, m2, m3 = m_modes
n1, n2, n3 = n_modes

m = m1 * m2 * m3
n = n1 * n2 * n3

s1 = m1 * n1
s2 = m2 * n2
s3 = m3 * n3

assert (m, n) == (8, 8)
assert (s1, s2, s3) == (4, 4, 4)


## 3. Dense weight を固定する

### 何のための処理か

全 rank で同じ $W$ を使い、rank 以外の違いを混ぜないようにします。

### `weight_dense`

1. **意味**: TT 近似する元の dense weight
2. **数式上の定義**: $W$
3. **shape**: $(8,8)$
4. **各軸の意味**: axis 0 = output、axis 1 = input
5. **何から計算されるか**: 固定 seed の乱数
6. **用途**: 全 rank の共通比較対象


In [54]:
weight_dense = torch.randn(
    m,
    n,
    dtype=dtype,
    device=device,
)

assert weight_dense.shape == (8, 8)
assert torch.linalg.matrix_norm(weight_dense, ord="fro") > 0.0


## 4. 3-core TT-matrix と requested rank

3-core TT-matrix の一般形は、

$$
G^{(1)}\in\mathbb R^{1\times m_1\times n_1\times r_1},
$$

$$
G^{(2)}\in\mathbb R^{r_1\times m_2\times n_2\times r_2},
$$

$$
G^{(3)}\in\mathbb R^{r_2\times m_3\times n_3\times1}.
$$

今回は共通 rank 上限 $r$ を指定します。

$$
\boxed{(r_0,r_1,r_2,r_3)=(1,r,r,1)}
$$

ただし、`requested_rank = r` は **各内部 bond に許す上限**です。

各 SVD で実際に保持する成分数は、

$$
\min(\text{requested rank},\ \text{そのSVDで利用可能な成分数})
$$

とします。

したがって、指定した rank と、生成された core shape から読める `actual_rank_vector` は概念上分けて記録します。

### `requested_rank`

1. **意味**: 各内部 bond に許す共通 rank 上限
2. **数式上の定義**: $r$
3. **shape**: Python int scalar
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: sweep 候補 $\{1,2,4\}$ から選ぶ
6. **用途**: TT-SVD 各段階で残す成分数の上限を指定する

### `actual_rank_vector`

1. **意味**: 実際に生成された3-coreの bond axis size
2. **数式上の定義**: $(1,r_1,r_2,1)$
3. **shape**: Python tuple、長さ4
4. **各軸の意味**: TT の boundary / internal bond sizes
5. **何から計算されるか**: `core1.shape[-1]`, `core2.shape[-1]`
6. **用途**: requested rank と実際の core shape を区別して記録する

`actual_rank_vector` は **core の軸サイズ**です。テンソルに必要な最小 TT-rank や、閾値で判定した数値的 rank とは混同しません。


## 5. 3-core TT-SVD helper

### 何のための処理か

既習の逐次 TT-SVD を helper として完成コードで用意します。今回の演習では、この helper を呼び出して rank sweep に集中します。

最初に dense weight を、

$$
(m_1,m_2,m_3,n_1,n_2,n_3)
$$

へ reshape し、physical pair が隣り合うように、

$$
(m_1,n_1,m_2,n_2,m_3,n_3)
$$

へ permute します。

その後、

$$
(s_1,s_2,s_3)=(4,4,4)
$$

として逐次 SVD を行います。

### 第1段階

$$
(4,4,4)
\rightarrow
A_1\in\mathbb R^{4\times16}.
$$

SVD 後、最大 `requested_rank` 成分を残し、左特異ベクトルから $G^{(1)}$ を作ります。

### 第2段階

第1段階の remainder は shape $(r_1, s_2s_3)$ です。これを

$$
A_2\in\mathbb R^{(r_1s_2)\times s_3}
$$

へ reshape して SVD します。

元テンソルを bond 2 で直接 unfold した構造上の shape は

$$
(s_1s_2,s_3)=(16,4)
$$

ですが、**逐次 TT-SVD の第2段階では第1段階の truncation 後なので**、実際の入力 shape は

$$
(r_1s_2,s_3)
$$

です。


In [55]:
def tt_svd_matrix_3core_with_rank_cap(
    weight: torch.Tensor,
    out_modes: tuple[int, int, int],
    in_modes: tuple[int, int, int],
    requested_rank: int,
):
    m1, m2, m3 = out_modes
    n1, n2, n3 = in_modes

    assert requested_rank >= 1
    assert weight.shape == (
        m1 * m2 * m3,
        n1 * n2 * n3,
    )

    s1 = m1 * n1
    s2 = m2 * n2
    s3 = m3 * n3

    # (m, n)
    # -> (m1, m2, m3, n1, n2, n3)
    weight_6d = weight.reshape(
        m1, m2, m3,
        n1, n2, n3,
    )

    # (m1, m2, m3, n1, n2, n3)
    # -> (m1, n1, m2, n2, m3, n3)
    weight_interleaved = weight_6d.permute(
        0, 3, 1, 4, 2, 5
    )

    # (m1, n1, m2, n2, m3, n3)
    # -> (s1, s2, s3) = (4, 4, 4)
    tensor_3d = weight_interleaved.reshape(
        s1, s2, s3
    )

    # ---------- TT-SVD stage 1 ----------
    # (s1, s2, s3)
    # -> A1: (s1, s2*s3) = (4, 16)
    A1 = tensor_3d.reshape(s1, s2 * s3)

    # A1: (s1, s2*s3)
    # U1: (s1, q1), S1: (q1,), Vh1: (q1, s2*s3)
    U1, S1, Vh1 = torch.linalg.svd(
        A1,
        full_matrices=False,
    )

    available_rank1 = S1.numel()
    r1 = min(requested_rank, available_rank1)

    U1_r = U1[:, :r1]
    S1_r = S1[:r1]
    Vh1_r = Vh1[:r1, :]

    # U1_r: (s1, r1)
    # -> G1: (1, m1, n1, r1)
    core1 = U1_r.reshape(
        1, m1, n1, r1
    )

    # diag(S1_r) @ Vh1_r
    # shape: (r1, s2*s3)
    remainder1 = S1_r[:, None] * Vh1_r

    # ---------- TT-SVD stage 2 ----------
    # remainder1: (r1, s2*s3)
    # -> A2: (r1*s2, s3)
    A2 = remainder1.reshape(r1 * s2, s3)

    # A2: (r1*s2, s3)
    # U2: (r1*s2, q2), S2: (q2,), Vh2: (q2, s3)
    U2, S2, Vh2 = torch.linalg.svd(
        A2,
        full_matrices=False,
    )

    available_rank2 = S2.numel()
    r2 = min(requested_rank, available_rank2)

    U2_r = U2[:, :r2]
    S2_r = S2[:r2]
    Vh2_r = Vh2[:r2, :]

    # U2_r: (r1*s2, r2)
    # -> G2: (r1, m2, n2, r2)
    core2 = U2_r.reshape(
        r1, m2, n2, r2
    )

    # diag(S2_r) @ Vh2_r: (r2, s3)
    # -> G3: (r2, m3, n3, 1)
    core3 = (S2_r[:, None] * Vh2_r).reshape(
        r2, m3, n3, 1
    )

    return core1, core2, core3


## 6. Dense reconstruction helper

### 何のための処理か

3個の core の内部 bond を縮約して、TT 表現から dense weight を復元します。

まず、

$$
(m_1,n_1,m_2,n_2,m_3,n_3)
$$

の軸順でテンソルを作ります。

その後、

$$
(m_1,m_2,m_3,n_1,n_2,n_3)
$$

へ戻し、最後に $(8,8)$ へ reshape します。

TT-SVD 側の interleave と完全に逆向きの処理になっていることが重要です。


In [56]:
def reconstruct_tt_matrix_3core(
    core1: torch.Tensor,
    core2: torch.Tensor,
    core3: torch.Tensor,
) -> torch.Tensor:
    _, m1, n1, r1 = core1.shape
    r1_b, m2, n2, r2 = core2.shape
    r2_b, m3, n3, _ = core3.shape

    assert r1 == r1_b
    assert r2 == r2_b

    # G1: (1, m1, n1, r1) -> (m1, n1, r1)
    c1 = core1[0]

    # G2: (r1, m2, n2, r2)
    c2 = core2

    # G3: (r2, m3, n3, 1) -> (r2, m3, n3)
    c3 = core3[..., 0]

    # Internal bonds r1, r2 を縮約する。
    # (m1,n1,r1), (r1,m2,n2,r2), (r2,m3,n3)
    # -> (m1,n1,m2,n2,m3,n3)
    weight_interleaved = torch.einsum(
        "ija,aklb,bmn->ijklmn",
        c1,
        c2,
        c3,
    )

    # (m1,n1,m2,n2,m3,n3)
    # -> (m1,m2,m3,n1,n2,n3)
    weight_6d = weight_interleaved.permute(
        0, 2, 4, 1, 3, 5
    )

    # (m1,m2,m3,n1,n2,n3)
    # -> (m1*m2*m3, n1*n2*n3) = (8,8)
    weight_dense_reconstructed = weight_6d.reshape(
        m1 * m2 * m3,
        n1 * n2 * n3,
    )

    return weight_dense_reconstructed


## 7. Rank candidates

今回は、

$$
\boxed{r\in\{1,2,4\}}
$$

を指定します。

第1 bond の元の unfolding は $(4,16)$、第2 bond の元の unfolding は $(16,4)$ なので、exact representation に必要な内部 rank の上限はいずれも4です。

### `rank_candidates`

1. **意味**: sweep する requested rank の候補
2. **数式上の定義**: $\{1,2,4\}$
3. **shape**: Python list、長さ3
4. **各軸の意味**: 各要素が1つの実験条件
5. **何から計算されるか**: 今回は固定
6. **用途**: 同一 weight / tensorization の下で rank だけを変える


In [57]:
rank_candidates = [1, 2, 4]
probe_rank = 2

assert rank_candidates == [1, 2, 4]
assert probe_rank in rank_candidates


## 8. Dense parameter count

### 何のための処理か

まず dense weight が何要素を保存しているかを基準として数えます。bias は含めません。

$$
\boxed{
N_{\mathrm{dense}}=8\times8=64
}
$$

### `dense_weight_params`

1. **意味**: dense weight の保存要素数
2. **数式上の定義**: $N_{\mathrm{dense}}=mn$
3. **shape**: Python int scalar
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: $m,n$
6. **用途**: TT の保存量と比較する基準


In [58]:
# TODO 1:
# dense weight の parameter count を計算してください。
dense_weight_params = m*n


### ヒント

今回は weight だけなので、

$$
N_{\mathrm{dense}}=mn
$$

です。


In [59]:
assert isinstance(dense_weight_params, int)
assert dense_weight_params == weight_dense.numel()
assert dense_weight_params == 64


## 9. TT parameter count

### 何のための処理か

TT 表現で実際に保存する core 要素数を数えます。

一般には、

$$
\boxed{
N_{\mathrm{TT}}
=
\sum_{k=1}^{d}
r_{k-1}m_kn_kr_k
}
$$

です。

3-core で rank vector が $(1,r,r,1)$ の場合、各 core は、

$$
G^{(1)}:(1,2,2,r)
\quad\Rightarrow\quad
1\cdot2\cdot2\cdot r=4r,
$$

$$
G^{(2)}:(r,2,2,r)
\quad\Rightarrow\quad
r\cdot2\cdot2\cdot r=4r^2,
$$

$$
G^{(3)}:(r,2,2,1)
\quad\Rightarrow\quad
r\cdot2\cdot2\cdot1=4r.
$$

したがって、

$$
\boxed{
N_{\mathrm{TT}}(r)=4r+4r^2+4r=8r+4r^2
}
$$

です。

これは **保存する要素数**です。Gauge freedom を差し引いた独立自由度の数ではありません。

### `tt_params_formula_probe`

1. **意味**: `probe_rank` を $(1,r,r,1)$ と仮定したときの TT 保存要素数
2. **数式上の定義**: $8r+4r^2$
3. **shape**: Python int scalar
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: core shape と `probe_rank`
6. **用途**: core の `numel()` 合計と一致するか確認する


In [60]:
# TODO 2:
# probe_rank に対する TT parameter count を、
# 3個の core shape の式から計算してください。
tt_params_formula_probe = m1*n1*probe_rank + m2*n2*probe_rank**2 + m3*n3*probe_rank


### ヒント

$$
N_{\mathrm{TT}}(r)
=
4r+4r^2+4r
$$

です。


In [61]:
assert isinstance(tt_params_formula_probe, int)
assert tt_params_formula_probe == 32


## 10. Compression ratio

### 何のための処理か

保存量を dense weight と同じ基準で比較します。

$$
\boxed{
C(r)
=
\frac{N_{\mathrm{dense}}}{N_{\mathrm{TT}}(r)}
}
$$

この Notebook では、

- $C>1$: parameter count の意味で圧縮
- $C=1$: 同じ保存要素数
- $C<1$: TT の方が保存要素数が多い

と解釈します。

**compression ratio と推論速度は同じものではありません。**

### `compression_ratio_probe`

1. **意味**: probe rank での dense / TT 保存量比
2. **数式上の定義**: $C=N_{\mathrm{dense}}/N_{\mathrm{TT}}$
3. **shape**: Python float scalar
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: dense params と TT params
6. **用途**: parameter count 上の圧縮度合いを比較する


In [62]:
# TODO 3:
# probe_rank の compression ratio を計算してください。
compression_ratio_probe = dense_weight_params/tt_params_formula_probe


### ヒント

`dense params / TT params` です。


In [63]:
assert isinstance(compression_ratio_probe, float)
assert_close(
    torch.tensor(compression_ratio_probe, dtype=dtype),
    torch.tensor(2.0, dtype=dtype),
    rtol=rtol,
    atol=atol,
)


## 11. Helper を使って1つの TT approximation を作る

### 何のための処理か

`probe_rank = 2` を上限として3-core TT-SVDを行い、coreから dense weight を再構成します。

### `weight_tt_probe`

1. **意味**: probe rank で作った TT approximation の dense reconstruction
2. **数式上の定義**: $\widetilde W(r)$
3. **shape**: $(8,8)$
4. **各軸の意味**: axis 0 = output、axis 1 = input
5. **何から計算されるか**: 3個の TT core
6. **用途**: 元の $W$ との reconstruction error を測る


In [64]:
# TODO 4:
# helper を使って probe_rank の3-core TT approximationを作り、
# dense weightへ再構成してください。
core1_probe, core2_probe, core3_probe = tt_svd_matrix_3core_with_rank_cap(weight_dense,m_modes,n_modes,probe_rank)
weight_tt_probe = reconstruct_tt_matrix_3core(core1_probe, core2_probe, core3_probe)
actual_rank_vector_probe = (
    1,
    core1_probe.shape[-1],
    core2_probe.shape[-1],
    1,
)


### ヒント

- `tt_svd_matrix_3core_with_rank_cap(...)`
- `reconstruct_tt_matrix_3core(...)`

を使います。

actual rank vector は core shape から、

$$
(1,r_1,r_2,1)
$$

として読めます。


In [65]:
assert weight_tt_probe.shape == weight_dense.shape
assert actual_rank_vector_probe == (
    1,
    core1_probe.shape[-1],
    core2_probe.shape[-1],
    1,
)

# 今回の固定設定では requested rank=2 が両 bond で利用可能。
assert actual_rank_vector_probe == (1, 2, 2, 1)

# core の実保存量と shape 式が一致することを確認。
tt_params_numel_probe = (
    core1_probe.numel()
    + core2_probe.numel()
    + core3_probe.numel()
)
assert tt_params_numel_probe == tt_params_formula_probe


## 12. Reconstruction error

### 何のための処理か

rank 制約によって $W$ をどれだけ正確に表現できたかを Frobenius norm で測ります。

absolute error:

$$
\boxed{
E_F(r)
=
\|W-\widetilde W(r)\|_F
}
$$

relative error:

$$
\boxed{
E_{\mathrm{rel}}(r)
=
\frac{\|W-\widetilde W(r)\|_F}{\|W\|_F}
}
$$

今回は $W\neq0$ を確認済みなので、分母に小さい定数は足しません。

### `absolute_error_probe`

1. **意味**: absolute Frobenius reconstruction error
2. **数式上の定義**: $\|W-\widetilde W(r)\|_F$
3. **shape**: scalar tensor
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: dense weight と reconstructed weight
6. **用途**: 再構成誤差の絶対量を測る

### `relative_error_probe`

1. **意味**: relative Frobenius reconstruction error
2. **数式上の定義**: $E_F(r)/\|W\|_F$
3. **shape**: scalar tensor
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: absolute error と $\|W\|_F$
6. **用途**: weight scale を基準化して rank 間を比較する


In [66]:
# TODO 5:
# probe_rank の absolute / relative Frobenius reconstruction error を
# 計算してください。
absolute_error_probe = torch.linalg.matrix_norm(weight_dense - weight_tt_probe,ord="fro")
relative_error_probe = absolute_error_probe/torch.linalg.matrix_norm(weight_dense,ord="fro")


### ヒント

`torch.linalg.matrix_norm(..., ord="fro")` を使えます。


In [67]:
weight_dense_fro = torch.linalg.matrix_norm(
    weight_dense,
    ord="fro",
)

assert weight_dense_fro > 0.0
assert absolute_error_probe.shape == torch.Size([])
assert relative_error_probe.shape == torch.Size([])
assert absolute_error_probe >= 0.0
assert relative_error_probe >= 0.0

assert_close(
    relative_error_probe,
    absolute_error_probe / weight_dense_fro,
    rtol=rtol,
    atol=atol,
)


## 13. TT-rank sweep

### 何のための処理か

同じ $W$ と同じ tensorization に対し、requested rank だけを変えます。

各 rank で記録するものは、

- `requested_rank`
- `actual_rank_vector`
- TT params
- compression ratio
- absolute Frobenius reconstruction error
- relative Frobenius reconstruction error
- parameter count の意味で圧縮か

です。

### `records`

1. **意味**: rank ごとの評価値をまとめた実験記録
2. **数式上の定義**: 各 $r$ に対する保存量・圧縮率・再構成誤差
3. **shape**: Python list of dict、長さ3
4. **各軸の意味**: 各 list 要素が1つの requested rank 条件
5. **何から計算されるか**: 各 rank の TT cores と reconstruction
6. **用途**: rank trade-off を DataFrame で比較する


In [68]:
records = []

for requested_rank in rank_candidates:
    # TODO 6:
    # 各 requested rank について、
    # 1) 3-core TT-SVD
    # 2) dense reconstruction
    # 3) actual rank vector
    # 4) core numel 合計による TT params
    # 5) shape式による TT params
    # 6) compression ratio
    # 7) absolute / relative Frobenius error
    # 8) parameter count上の圧縮判定
    # を計算して records に追加してください。

    core1_r, core2_r, core3_r = tt_svd_matrix_3core_with_rank_cap(
        weight_dense, m_modes, n_modes, requested_rank
    )
    weight_tt_r = reconstruct_tt_matrix_3core(core1_r, core2_r, core3_r)

    actual_rank_vector_r = (
        1,
        core1_r.shape[-1],
        core2_r.shape[-1],
        1,
    )
    _, r1, r2, _ = actual_rank_vector_r

    tt_params_r = core1_r.numel() + core2_r.numel() + core3_r.numel()
    tt_params_formula_r = m1 * n1 * r1 + r1 * m2 * n2 * r2 + r2 * m3 * n3

    compression_ratio_r = dense_weight_params / tt_params_r

    absolute_error_r = torch.linalg.matrix_norm(
        weight_dense - weight_tt_r, ord="fro"
    )
    relative_error_r = absolute_error_r / weight_dense_fro

    parameter_compressed_r = tt_params_r < dense_weight_params

    # 実際の core 要素数と、actual rank vector を使った式が一致すること。
    assert tt_params_r == tt_params_formula_r

    records.append(
        {
            "requested_rank": requested_rank,
            "actual_rank_vector": actual_rank_vector_r,
            "TT params": tt_params_r,
            "compression ratio": compression_ratio_r,
            "absolute Frobenius reconstruction error": absolute_error_r.item(),
            "relative Frobenius reconstruction error": relative_error_r.item(),
            "parameter count compressed": parameter_compressed_r,
        }
    )


### ヒント

actual rank vector を

$$
(1,r_1,r_2,1)
$$

とすると、shape 式による保存量は、

$$
1\cdot m_1n_1r_1
+
r_1m_2n_2r_2
+
r_2m_3n_3\cdot1
$$

です。

relative error の分母は、全 rank 共通の `weight_dense_fro` です。


In [69]:
assert len(records) == len(rank_candidates)
assert [row["requested_rank"] for row in records] == rank_candidates

expected_tt_params = {
    1: 12,
    2: 32,
    4: 96,
}

expected_compression_ratio = {
    1: 64 / 12,
    2: 2.0,
    4: 64 / 96,
}

for row in records:
    r = row["requested_rank"]

    # この固定shapeでは requested rank がそのまま core axis size として使える。
    assert row["actual_rank_vector"] == (1, r, r, 1)

    assert row["TT params"] == expected_tt_params[r]

    assert math.isclose(
        row["compression ratio"],
        expected_compression_ratio[r],
        rel_tol=1e-12,
        abs_tol=1e-12,
    )

    assert row["parameter count compressed"] == (
        row["TT params"] < dense_weight_params
    )


## 14. Sweep 結果を DataFrame で表示する

結果表の表示は完成コードです。


In [70]:
results_df = pd.DataFrame(records)
results_df


,requested_rank,actual_rank_vector,TT params,compression ratio,absolute Frobenius reconstruction error,relative Frobenius reconstruction error,parameter count compressed
0,1,"(1, 1, 1, 1)",12,5.333333,7.201489e+00,8.298814e-01,True
1,2,"(1, 2, 2, 1)",32,2.000000,5.420826e+00,6.246823e-01,True
2,4,"(1, 4, 4, 1)",96,0.666667,6.301471e-15,7.261656e-16,False


## 15. 保存量と compression ratio の関係を確認する

### 何のための処理か

今回の固定 mode size では、requested rank が大きくなるほど core の保存要素数が増えます。

したがって、

$$
N_{\mathrm{TT}}\uparrow
\quad\Longrightarrow\quad
C=\frac{N_{\mathrm{dense}}}{N_{\mathrm{TT}}}\downarrow
$$

となることを結果から確認します。

### `params_increase_with_rank`

1. **意味**: rank 条件の増加に伴い TT params が増えているか
2. **数式上の定義**: 隣接条件で $N_{TT}(r_i)<N_{TT}(r_{i+1})$
3. **shape**: Python bool
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: `records`
6. **用途**: rank と保存量の関係を確認する

### `compression_decreases_with_rank`

1. **意味**: rank 条件の増加に伴い compression ratio が下がるか
2. **数式上の定義**: 隣接条件で $C(r_i)>C(r_{i+1})$
3. **shape**: Python bool
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: `records`
6. **用途**: 保存量増加と圧縮率低下の関係を確認する


In [71]:
# TODO 7:
# sweep結果から、
# 1) requested rank が上がると TT params が増えること
# 2) compression ratio が下がること
# を確認してください。
params_increase_with_rank = all(
    a["TT params"] < b["TT params"]
    for a, b in zip(records, records[1:])
)
compression_decreases_with_rank = all(
    a["compression ratio"] > b["compression ratio"]
    for a, b in zip(records, records[1:])
)


### ヒント

隣り合う `records` を比較します。`zip(...)` と `all(...)` を使って構いません。


In [72]:
assert bool(params_increase_with_rank)
assert bool(compression_decreases_with_rank)


## 16. Reconstruction error と「最良近似」を混同しない

### 何のための確認か

2-core の1回の truncated SVD とは異なり、3-core TT-SVD は **逐次 truncation** です。

rank 上限を増やせば許される TT 表現の候補集合は広がるので、理論上の **最良 TT 近似誤差**

$$
E_{\mathrm{best}}(r)
$$

は悪化しません。

一方、通常の逐次 TT-SVD が返す近似を

$$
E_{\mathrm{TTSVD}}(r)
$$

とすると、TT-SVD は一般に global best TT approximation そのものではありません。

既習の準最良性は、rank vector を固定したとき、

$$
\boxed{
E_{\mathrm{best}}(r)
\le
E_{\mathrm{TTSVD}}(r)
\le
\sqrt{d-1}\,E_{\mathrm{best}}(r)
}
$$

です。

今回は $d=3$ なので、

$$
\boxed{
E_{\mathrm{TTSVD}}(r)
\le
\sqrt{2}\,E_{\mathrm{best}}(r)
}
$$

となります。

ただし、**この保証だけから、今回実測した TT-SVD error が requested rank に対して単調非増加だとは結論しません。**

`E_best(r)` はこの sweep では計算していない理論上の比較基準です。

下では実測誤差がどう並んだかを **観察するだけ**にし、単調性を一般的な必須 assert にはしません。


In [73]:
observed_errors = [
    row["absolute Frobenius reconstruction error"]
    for row in records
]

observed_error_nonincreasing = all(
    next_error <= current_error + 1e-12
    for current_error, next_error in zip(
        observed_errors,
        observed_errors[1:],
    )
)

print("observed errors:", observed_errors)
print(
    "observed non-increasing in this experiment:",
    observed_error_nonincreasing,
)

# NOTE:
# ここでは observed_error_nonincreasing を assert しない。
# これは今回の実験結果の観察であり、
# 一般の multi-core TT-SVD に対する必須性質として扱わない。


observed errors: [7.201489362626453, 5.420826316238032, 6.301470836549541e-15]
observed non-increasing in this experiment: True


## 17. requested rank = 4 では exact reconstruction を確認する

### 何のための確認か

今回の元テンソル $(4,4,4)$ に対し、2つの bond cut の unfolding は、

$$
(4,16),
\qquad
(16,4)
$$

です。

したがって内部 rank 上限4を許せば、各段階で利用可能な SVD 成分をすべて保持できます。

そのため requested rank 4 では、浮動小数点誤差の範囲で、

$$
\widetilde W(4)\approx W
$$

を確認します。

一方、保存要素数は、

$$
N_{TT}(4)
=8\cdot4+4\cdot4^2
=96
$$

なので、

$$
96>64=N_{dense}.
$$

したがって、**ほぼ完全再構成できても parameter count の意味では圧縮ではありません。**


In [74]:
core1_full, core2_full, core3_full = tt_svd_matrix_3core_with_rank_cap(
    weight=weight_dense,
    out_modes=m_modes,
    in_modes=n_modes,
    requested_rank=4,
)

weight_tt_full = reconstruct_tt_matrix_3core(
    core1_full,
    core2_full,
    core3_full,
)

assert_close(
    weight_tt_full,
    weight_dense,
    rtol=rtol,
    atol=atol,
)

full_rank_params = (
    core1_full.numel()
    + core2_full.numel()
    + core3_full.numel()
)

assert full_rank_params == 96
assert full_rank_params > dense_weight_params


## 18. まとめ

今回の3-core sweepでは、tensorizationを固定して、requested rank

$$
r\in\{1,2,4\}
$$

だけを変更しました。

保存量は、actual rank vector が $(1,r,r,1)$ のとき、

$$
\boxed{
N_{TT}(r)=8r+4r^2
}
$$

です。

そのため今回の固定設定では、

| requested rank | TT params | compression ratio |
|---:|---:|---:|
| 1 | 12 | $64/12$ |
| 2 | 32 | $2$ |
| 4 | 96 | $64/96$ |

となります。

同時に各 rank で、

$$
\|W-\widetilde W(r)\|_F
$$

と、

$$
\frac{\|W-\widetilde W(r)\|_F}{\|W\|_F}
$$

を記録します。

ここで確認するのは、

$$
\boxed{
\text{rank}
\rightarrow
\text{保存量 / compression ratio}
}
$$

と、

$$
\boxed{
\text{rank}
\rightarrow
\text{weight reconstruction error}
}
$$

の trade-off です。

今回は weight representation までで終了します。prediction / accuracy には進みません。
